# M04A: Instructions & Conversation Chaining

Two parameters unlock multi-turn AI: `instructions` for behavior, `previous_response_id` for memory.

**Topics:**
- Simple chatbot with memory
- Multi-turn task assistant
- ConversationManager class

---

## 🔧 Step 1: Setup

**Note:** In this module, we use raw `client.responses.create()` calls so you can see exactly how `instructions` and `previous_response_id` work. Later **in this notebook**, we'll build a `ConversationManager` class to handle the complexity.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


# Display helper for long outputs
def truncate_response(text, max_length=1200):
    """Truncate text for cleaner display."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


print(f"✅ Setup complete: Using {MODEL}!")

---

## 📝 The Instructions Parameter

The `instructions` parameter guides AI behavior — like system messages in Chat Completions.

In [ ]:
prompt = "Explain quantum computing"

# Example 1: No instructions (default behavior)
print("🤖 WITHOUT INSTRUCTIONS")
print("="*60)
response = client.responses.create(
    model=MODEL,
    input=prompt
)
print(truncate_response(response.output_text))


# Example 2: With instructions (teacher role)
print("\n🎓 WITH INSTRUCTIONS (Teacher Role)")
print("="*60)
response = client.responses.create(
    model=MODEL,
    input=prompt,
    instructions="You are a patient teacher explaining to a 10-year-old. Use simple words. Be concise."
)
print(truncate_response(response.output_text))


# Example 3: Different instructions (technical role)
print("\n💻 WITH INSTRUCTIONS (Technical Role)")
print("="*60)
response = client.responses.create(
    model=MODEL,
    input=prompt,
    instructions="You are a researcher. Be technical, precise, and use proper terminology. Be concise."
)
print(truncate_response(response.output_text))

### 💡 Key Observation

Same prompt, different instructions → completely different responses.

---

## 📋 Best Practices for Instructions

**✅ DO:**
- Be specific about role/persona
- Include output format requirements
- Specify constraints (add "be concise")

**❌ DON'T:**
- Make instructions too long (wastes tokens)
- Include the actual query in instructions
- Expect instructions to persist (not reliable)

---

## 🔗 Multi-turn Conversations with previous_response_id

Instead of sending the entire conversation history each time, just link to the previous response.
```
Turn 1 ◄── Turn 2 ◄── Turn 3 ◄── Turn 4

```

Each response ID contains a reference to its predecessor — the API follows the chain back automatically.

**How it works:**
1. First turn: No `previous_response_id`
2. Second turn: Pass the ID from first response
3. Third turn: Pass the ID from second response

In [ ]:
print("💬 MULTI-TURN CONVERSATION DEMO")
print("="*60)

instructions = "You are a friendly assistant. Remember details. Be concise."

# Turn 1
print("\n👤 User: Hi, my name is Alex")
response1 = client.responses.create(
    model=MODEL,
    input="Hi, my name is Alex",
    instructions=instructions
)
print(f"🤖 Assistant: {truncate_response(response1.output_text)}")
print(f"\n📝 Saved response ID: {response1.id[:20]}...")

# Turn 2
print("\n👤 User: What's my name?")
response2 = client.responses.create(
    model=MODEL,
    input="What's my name?",
    instructions=instructions,
    previous_response_id=response1.id
)
print(f"🤖 Assistant: {truncate_response(response2.output_text)}")

# Turn 3
print("\n👤 User: What did I first say to you?")
response3 = client.responses.create(
    model=MODEL,
    input="What did I first say to you?",
    instructions=instructions,
    previous_response_id=response2.id
)
print(f"🤖 Assistant: {truncate_response(response3.output_text)}")

print("="*60)

### 💡 Key Points

The assistant remembered context across all 3 turns — each turn links to the previous via `previous_response_id`.

Notice we passed `instructions` on every turn.

**Cost note:** Chaining means you send less data per request, but the model still uses prior turns as context — so longer chains typically cost more than shorter ones.

---

## ⚠️ Important: Instructions Don't Persist

Instructions apply only to the *current* response. If you don't pass them on later turns, behavior may drift — the model might continue the previous style from context, but it's not reliable. Pass instructions on every turn for consistent behavior.

In [ ]:
print("⚠️  INSTRUCTIONS DON'T PERSIST DEMO")
print("="*60)

pirate_instructions = "You are a pirate. Always talk like a pirate. Be concise."

# Turn 1: WITH instructions (pirate)
print("\nTurn 1: WITH instructions (pirate)")
response1 = client.responses.create(
    model=MODEL,
    input="Tell me about the weather in one sentence",
    instructions=pirate_instructions
)
print(f"Response: {truncate_response(response1.output_text)}")

# Turn 2: WITHOUT instructions
print("\nTurn 2: WITHOUT instructions (forgot to pass them)")
response2 = client.responses.create(
    model=MODEL,
    input="Tell me more",
    previous_response_id=response1.id
)
print(f"Response: {truncate_response(response2.output_text)}")

# Turn 3: WITH instructions again (pirate)
print("\nTurn 3: WITH instructions again (pirate)")
response3 = client.responses.create(
    model=MODEL,
    input="And one more thing",
    instructions=pirate_instructions,
    previous_response_id=response2.id
)
print(f"Response: {truncate_response(response3.output_text)}")

print("="*60)

### 💡 Critical Lesson

**Turn 1:** Pirate voice (instructions provided)  
**Turn 2:** May drift (instructions missing — behavior unpredictable)  
**Turn 3:** Pirate voice (instructions restored)

Without instructions, the model may continue the previous style from context, or it may drift. Pass instructions on every turn for *consistent* behavior — or use `ConversationManager` 👇 to handle it automatically.

---

## 🏗️ Building a Conversation Manager

In [ ]:
class ConversationManager:
    """Manages multi-turn conversations with the Responses API."""
    
    def __init__(self, instructions="You are a helpful assistant. Be concise."):
        self.instructions = instructions  # Stored once, used every turn
        self.transcript = []
        self.last_response_id = None
    
    def send(self, message):
        """Send message and return response."""
        try:
            kwargs = {
                "model": MODEL,
                "input": message,
                "instructions": self.instructions
            }
            
            if self.last_response_id:
                kwargs["previous_response_id"] = self.last_response_id
            
            response = client.responses.create(**kwargs)
            
            # Save for next turn
            response_text = response.output_text.strip()
            self.last_response_id = response.id
            
            # Track for display
            self.transcript.append({"role": "user", "content": message})
            self.transcript.append({"role": "assistant", "content": response_text})
            
            return response_text
            
        except Exception as e:
            print(f"❌ API Error: {e}")
            return f"Error: {str(e)}"
    
    def show_transcript(self):
        """Display conversation transcript."""
        print("\n📜 CONVERSATION TRANSCRIPT")
        print("="*60)
        
        for turn in self.transcript:
            emoji = "👤" if turn["role"] == "user" else "🤖"
            role = turn["role"].title()
            content = truncate_response(turn["content"])
            print(f"\n{emoji} {role}: {content}")
        
        print("="*60)
    
    def reset(self):
        """Start a new conversation."""
        self.transcript = []
        self.last_response_id = None
        print("✅ Conversation reset")


# --------------------------------------------------------------
print("✅ ConversationManager class ready!")

### Test the Conversation Manager

In [ ]:
conversation = ConversationManager(
    instructions="You are a friendly assistant. Answer in one sentence. Be concise."
)

print("💬 TESTING CONVERSATION MANAGER")
print("="*60)

# Turn 1
print("\n👤 User: My favorite color is blue")
response1 = conversation.send("My favorite color is blue")
print(f"🤖 Assistant: {truncate_response(response1)}")

# Turn 2
print("\n👤 User: What's my favorite color?")
response2 = conversation.send("What's my favorite color?")
print(f"🤖 Assistant: {truncate_response(response2)}")

# Turn 3
print("\n👤 User: What did I tell you in the first message?")
response3 = conversation.send("What did I tell you in the first message?")
print(f"🤖 Assistant: {truncate_response(response3)}")

conversation.show_transcript()

### 💡 Key Benefit

`ConversationManager` tracks response IDs and applies instructions on every turn — so you don't have to.

---

### 💪 Your Turn: Build a Task Assistant

Build a multi-turn assistant that helps break down complex tasks.

**Requirements:**
1. User describes a project/task
2. Assistant breaks it into steps
3. User can ask follow-up questions
4. Assistant remembers the original task

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Build a Task Assistant
# --------------------------------------------------------------
# Objective: Use ConversationManager to build a planner that remembers context.

task_assistant = ConversationManager(
    instructions="""You are a project planning assistant.
1. Break tasks into clear, numbered steps.
2. Be concise and actionable.
3. If the task is vague, ask clarifying questions."""
)

print("🎯 TASK ASSISTANT")
print("="*60)

# TODO 1: Send a project description


# TODO 2: Ask a follow-up question


# TODO 3: Test memory with a recall question


# TODO 4: Show full transcript

---

## 🎯 Key Takeaways

### What You Learned

**📝 Instructions Parameter:**
- Sets the role, tone, and constraints for each response
- Must be passed on every request (does not persist)
- Replaces repetitive system messages in conversation history

**🔗 Conversation Chaining:**
- Use `previous_response_id` to link turns together
- The API manages context history server-side
- Send less data per request (just the new message)

**🔑 Best Practices:**
- Always pass instructions for consistent behavior across turns
- Track response IDs carefully; losing an ID breaks the chain
- Use `ConversationManager` to abstract the ID tracking

### Quick Reference

**The Flow:** Create client → Send with `instructions` → Save `response.id` → Pass as `previous_response_id` on next turn

---

### 📍 Next Step

**M04B: Role-Based Prompts & Personas** — Effective personas and combining roles with the instructions parameter.

---

## 🔧 Troubleshooting

**Conversation losing context?**
- Check you're passing `previous_response_id`
- Verify response IDs are being saved correctly
- Make sure you're chaining from the most recent response

**Behavior inconsistent across turns?**
- If you don't pass instructions each turn, behavior may drift
- Pass instructions every turn for consistent behavior
- Check your `ConversationManager` is applying instructions

**Response ID errors?**
- Make sure response ID exists before using it
- Check for None values
- Verify API call succeeded before saving ID

**Responses too long?**
- Add "be concise" to instructions
- Request specific word/sentence limits
- Use `truncate_response()` helper to truncate for display

**How long can conversation chains be?**
- API manages context window automatically
- Older turns drop when needed (model may forget early parts)
- Consider summarizing after 20+ turns

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---